# CUDA C++ reference kernels: do checker verdicts depend on the reference?

> Run on Kaggle (Tesla T4). Result: `results/refcheck_t4.json`, 0 of 80 candidates change verdict; see `docs/cuda_reference.md`.
> The CPU counterpart is `results/refcheck_cpu.json` (`python -m triton_equiv.refcheck` in WSL).

1. Builds `csrc/ref_cuda.cu` (naive CUDA: one thread per output, sequential loops, no tensor cores) and
   `csrc/ref_cpu.cpp`. Both run the same `csrc/ref_kernels.h`. Then checks that the two builds agree on
   every kernel and size, and that both agree with PyTorch.
2. Runs the standard checker on every reference kernel and every mutant in real Triton on the T4, against
   three references: PyTorch, the CUDA kernels in fp32, and the CUDA kernels in fp64. Lists every
   candidate whose verdict changes with the reference.

In [ ]:
# Kaggle: Settings -> Accelerator -> GPU T4. Internet must be on.
import glob, os, shutil, subprocess, sys
os.environ["TRITON_INTERPRET"] = "0"
found = glob.glob("/kaggle/input/**/triton_equiv", recursive=True)
if found:   # a private dataset holding src/triton_equiv at a given commit (see SOURCE.txt)
    DS = found[0].rsplit("/", 1)[0]
    print("dataset at", DS, open(DS + "/SOURCE.txt").read() if os.path.exists(DS + "/SOURCE.txt") else "")
    shutil.copytree(DS, "/kaggle/working/tileproof_src", dirs_exist_ok=True)
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/rajul-kk/triton_tileproof.git", "/kaggle/working/tp"], check=True)
    shutil.copytree("/kaggle/working/tp/src", "/kaggle/working/tileproof_src", dirs_exist_ok=True)
    shutil.copy("/kaggle/working/tp/results/labels.json", "/kaggle/working/labels.json")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "z3-solver", "ninja",
                "git+https://github.com/rajul-kk/tinytail.git"], check=True)
sys.path[:0] = ["/kaggle/working/tileproof_src"]

In [ ]:
import json, torch, triton
print(torch.__version__, triton.__version__, torch.cuda.get_device_name(0))
from triton_equiv import cuda_ref, harness, refcheck
from triton_equiv.kernels import KERNELS
for b in cuda_ref.Backend:
    print(b, cuda_ref.build_error(b) or "built")

## Control: the CUDA build against the CPU build and against PyTorch

In [ ]:
worst = {}
for name, spec in KERNELS.items():
    for size in [s for ss in spec.checker_sizes.values() for s in ss]:
        L = spec.make_launch(**size)
        x = harness.make_inputs(L, seed=1)
        (out,) = L.outputs
        for acc in cuda_ref.Acc:
            g = cuda_ref.cpp_reference(name, L, x, acc, cuda_ref.Backend.CUDA)[out].double()
            c = cuda_ref.cpp_reference(name, L, x, acc, cuda_ref.Backend.CPU)[out].double()
            worst[(name, acc.value)] = max(worst.get((name, acc.value), 0.0), float((g - c).abs().max()))
        t = harness.torch_reference(name, L, x)[out]
        g32 = cuda_ref.cpp_reference(name, L, x, cuda_ref.Acc.F32, cuda_ref.Backend.CUDA)[out]
        assert harness.close(g32, t, L.dtype(out), "kernelbench"), (name, size)
for k, v in worst.items():
    print(k, f"cuda vs cpu build max |diff| = {v:.3g}")

## Checker verdicts under three references, real Triton on the T4

In [ ]:
res = refcheck.main(["--device", "cuda", "--trials", "3", "--labels", "/kaggle/working/labels.json"
                     if os.path.exists("/kaggle/working/labels.json") else "/kaggle/working/tileproof_src/labels.json",
                     "--out", "/kaggle/working/refcheck_t4.json"])
print(json.dumps(res["reference_kernel_max_err"], indent=1))
res["disagreements"]